# **Deductive Logic in Language Models: Horizontal vs. Vertical Reasoning - Task 2 CoT vs. Curriculum Learning**

This notebook reproduces the experiments described in Section 8 of [D. Maltoni and M. Ferrara, *"Deductive Logic in Language Models: Horizontal vs. Vertical Reasoning"*, Machine Learning and Knowledge Extraction, 2026](https://doi.org/10.3390/make8070214).

The following code imports the modules and functions required to run the notebook.

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from dataset import GraphDataset_Splitted_Compact
from utils import train, get_loaders
from transformer_lens import HookedTransformer, HookedTransformerConfig

The cell below defines a utility function for visualizing the results hown in Figures 8 and 9 of the paper.

In [ ]:
def plot_accuracies(epochs, accuracies, filter_epochs=None, fontsize=14, figsize=(16, 6)):
    if filter_epochs is not None:
        indices = [i for i, e in enumerate(epochs) if e in filter_epochs]
        epochs = [epochs[i] for i in indices]
        accuracies = {label: [acc[i] for i in indices] for label, acc in accuracies.items()}

    labels = list(accuracies.keys())
    n_labels = len(labels)
    n_epochs = len(epochs)

    x = np.arange(n_epochs)
    bar_width = 0.8 / n_labels

    _, ax = plt.subplots(figsize=figsize)
    for i, label in enumerate(labels):
        offset = (i - n_labels / 2) * bar_width + bar_width / 2
        ax.bar(x + offset, accuracies[label], width=bar_width, label=label)

    ax.set_xlabel('Epoch', fontsize=fontsize)
    ax.set_ylabel('Accuracy (%)', fontsize=fontsize)
    ax.set_xticks(x)
    ax.set_xticklabels(epochs, fontsize=fontsize)
    ax.tick_params(axis='y', labelsize=fontsize)
    ax.grid(axis='y')
    plt.tight_layout()
    plt.show()

The following parameters configure the dataset and the nanoGPT architecture used in the experiment.

In [ ]:
# dataset configuration
dataset_config = {
    "n_states": 16,
    "dataset_file_name": "dataset_150k_remapped_paper.txt",  
    "dataset_depth_file_name": "dataset_150k_remapped_depths.txt",               
    "n_samples": 150_000,                                             
    "batch_size": 64,
}

# model configuration
model_config = {
    "n_layers": 6,
    "d_model": 128,
    "n_heads": 1,
    "d_mlp": 512,
    "d_head": 128,
}

init_seed = 72

Running the following cell properly initialises all random seeds to ensure reproducible results.

In [ ]:
random.seed(init_seed)
np.random.seed(init_seed)
torch.manual_seed(init_seed)
torch.cuda.manual_seed_all(init_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

The following code loads the dataset along with the root-to-leaf distance associated with each tree.

In [ ]:
dataset = GraphDataset_Splitted_Compact(dataset_config["n_states"], dataset_config["dataset_file_name"], dataset_config["dataset_depth_file_name"])

The following code creates the training and test data loaders using the batch size specified in the dataset configuration.

In [ ]:
train_loader, test_loader = get_loaders(dataset, dataset_config["batch_size"])

The following code configures and initialises the Transformer model.

In [ ]:
# setup model
cfg = HookedTransformerConfig(
    n_layers=model_config["n_layers"],
    d_model=model_config["d_model"],
    n_ctx=dataset.max_seq_length - 1,
    n_heads=model_config["n_heads"],
    d_mlp=model_config["d_mlp"],
    d_head=model_config["d_head"],
    d_vocab=len(dataset.idx2tokens),
    device="cuda",
    attention_dir="causal",
    act_fn="gelu",
)
model = HookedTransformer(cfg)

The following code trains the model and records its autoregressive accuracy on the training and test sets. It also tracks the examples correctly processed at different stages of training and groups them by their root-to-leaf distance.

In [ ]:
# start training loop
train_acc_ar, test_acc_ar,test_depth_sequences, test_token_distances = train(model,train_loader,test_loader,n_epochs=10,learning_rate=3e-4, eval_depth_info=True)

Executing the cell below will reproduce Figure 8.

In [ ]:
#Figure 8

epochs = []
accuracies = {f'{str(i)}': [] for i in range(len(test_token_distances[0][0]))}

for epoch in range(len(test_token_distances)):
    epochs.append(epoch + 1)
    token_dist_from_leaf, token_dist_from_leaf_count = test_token_distances[epoch]
    for dist in range(len(token_dist_from_leaf)):
        count = token_dist_from_leaf_count[dist]
        if count > 0:
            acc = token_dist_from_leaf[dist] / count
            accuracies[f'{str(dist)}'].append(100*acc)
            
plot_accuracies(epochs, accuracies, [1,2,10],figsize=(12, 5))

Executing the cell below will reproduce Figure 9.

In [ ]:
#Figure 9

epochs = []
accuracies = {k: [] for k in test_depth_sequences[0][0].keys()}

for epoch in range(len(test_depth_sequences)):
    epochs.append(epoch + 1)
    depth_seq_correct_count, depth_seq_count = test_depth_sequences[epoch]
    for key_depth in depth_seq_correct_count.keys():
        accuracies[key_depth].append(100*depth_seq_correct_count[key_depth] / depth_seq_count[key_depth])

plot_accuracies(epochs, accuracies, [1,2,10],figsize=(12, 5))
    